### LangChain을 활용한 ReAct 에이전트

### 1. LangChain Agent 프레임워크 소개
LangChain은 에이전트를 쉽게 구축할 수 있도록 추상화된 클래스들을 제공합니다. 최신 버전에서는 내부적으로 LangGraph 기반의 강력한 상태 관리 엔진을 사용합니다.
- **LLM/ChatModel**: 추론을 담당하는 엔진 (예: `ChatOpenAI`)
- **Tools**: 에이전트가 외부 세계와 상호작용할 수 있는 기능의 집합
- **create_agent**: LLM과 도구를 결합하여 알아서 ReAct 사이클(Thought -> Action -> Observation)을 수행하는 지능형 에이전트 객체를 반환합니다.

### 2. Custom Tool 정의
`@tool` 데코레이터를 이용해 아주 쉽게 파이썬 함수를 에이전트가 사용할 수 있는 도구로 만들 수 있습니다.

In [3]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from dotenv import load_dotenv
load_dotenv(override=True)

# Tool 정의(@tool 데코레이션 사용)
@tool
def search_knowlege(query:str) ->str:
    '''지식베이스를 검색합니다. 모르는 정보가 있을대 사용하세요'''
    kb = {
        'LangChain' : '랭체인(LangChain)은 대규모 언어 모델(LLM)을 활용하여 강력한 AI 애플리케이션을 쉽게 구축할 수 있도록 돕는 오픈소스 개발 프레임워크입니다. LLM이 외부 데이터에 접근하고, 다양한 도구를 사용하며, 대화의 맥락을 기억할 수 있게 연결해 줍니다.',
        'ReAct' : "React(리액트)는 크게 두 가지 의미로 사용됩니다. 일상 영어 단어로는 '반응하다'라는 뜻을 가지며, IT 분야에서는 페이스북(메타)이 개발한 사용자 인터페이스(UI) 구축용 자바스크립트 라이브러리를 의미합니다"
    }
    for key, val in kb.items():
        if key.lower() in query.lower():
            return val
    return '관련 정보를 찾을 수 없습니다.'
@tool
def string_length(text:str)->int:
    '''문자열 길이를 반환합니다. 텍스트의 글자 수를 알아야 할때 사용하세요'''
    return len(text)

@tool
def calculate_discount(price:float, discount_rate:float)->float:
    '''원래 가격(price)과 할인율(discount_rate, 예 20%면 0.2)을 입력받아서 할인가격을 계산합니다.'''
    return price * (1-discount_rate)
tools = [search_knowlege,string_length,calculate_discount]    

### 3. ReAct 에이전트 생성 및 실행
최신 LangChain에서는 `create_agent` 함수 하나로 도구와 모델을 바인딩한 완성형 에이전트를 만들 수 있습니다.

In [4]:
from langchain.agents import create_agent
# chat모델 초기화
llm = ChatOpenAI(model='gpt-5.4-nano',temperature=0)
# 에이전트 생성
agent = create_agent(llm,tools)

### 4. 에이전트 실행 및 추적
`invoke` 메서드를 호출하면 에이전트는 내부적으로 `Thought -> Action -> Observation` 과정을 거쳐 정답을 도출합니다. 결과는 메시지 리스트 형태로 반환됩니다.

In [ ]:
# 단일도구사용
response1 = agent.invoke({'message':[('user','15000원짜리 물건을 15% 할인받으면 얼마인가요')  ]})